# Restoring Historical Photographs — Notebook 2: Modeling Methods and Fine-Tuning

**AAI-521 Group 3** · Nolan Robbins · Sean McGowan · Jackson Kenyon

This notebook fine-tunes four pretrained Hugging Face models on the curated
FSA/OWI + DOCUMERICA training split (notebook 01) and tracks validation performance
against two references:

* the **classical baseline**: the degraded input, bicubic upsampling, the grey image, or
  OpenCV Telea inpainting;
* the **zero-shot** pretrained model (step 0, before any update).

Set `RUN_TRAINING = True` to train (Colab A100 or a ≥ 16 GB GPU). Otherwise the cells read
the logged results from the checkpoint directory. Runs are resumable: re-running a cell
after a disconnect continues from `last_state.pt`.

## Environment setup

Runs unchanged on **Google Colab** or locally.

* **Colab:** clones the repo, installs it, mounts Google Drive and copies the packed
  dataset (`MyDrive/aai521/aai521_dataset.tar`, made by `scripts/pack_dataset.py`) to the
  VM's fast local disk. Checkpoints are written to `MyDrive/aai521/checkpoints`, so they
  survive disconnects.
* **Locally:** uses `$AAI521_DATA_ROOT` from `.env`.

All project logic lives in the importable `restoration` package (`src/restoration`), so
the notebooks stay focused on the narrative, results and figures.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/NolanRobbins/AAI-521-Final-Project.git /content/AAI-521-Final-Project 2>/dev/null || git -C /content/AAI-521-Final-Project pull -q
    %cd /content/AAI-521-Final-Project
    !pip install -q -r requirements.txt && pip install -q -e .
    sys.path.insert(0, "/content/AAI-521-Final-Project/src")  # editable install, no restart needed

from restoration import colab

paths = colab.setup()  # must run before importing model code (sets HF_HOME on Colab)
paths

In [ ]:
import os
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

from restoration.config import CONFIG_DIR, checkpoint_root
from restoration.evaluation.plotting import apply_style, plot_training_run
from restoration.training.sd_inpaint import InpaintLoraConfig, train_inpaint_lora
from restoration.training.trainer import TrainConfig, train

apply_style()
RUN_TRAINING = False  # True to (re)train; runs resume from their last checkpoint
NUM_WORKERS = min(8, os.cpu_count() or 2)


def load_config(name: str):
    path = CONFIG_DIR / "train" / name
    task = yaml.safe_load(path.read_text())["task"]
    model = InpaintLoraConfig if task == "inpaint" else TrainConfig
    return model.from_yaml(path, num_workers=NUM_WORKERS)


def run_task(cfg):
    return train_inpaint_lora(cfg) if cfg.task == "inpaint" else train(cfg)


def show_run(run_name: str, metrics=("psnr", "lpips"), title=None):
    csv = checkpoint_root() / run_name / "metrics.csv"
    if not csv.is_file():
        print(f"No results yet for {run_name} (set RUN_TRAINING = True)")
        return
    plot_training_run(csv, metrics=metrics, title=title)
    frame = pd.read_csv(csv)
    display(frame[frame["kind"] != "train"].drop(columns=["loss", "lr"], errors="ignore").dropna(axis=1, how="all").round(4).tail(4))

## 3. Modeling methods

### Transfer-learning strategy

Paired old-photo data is scarce, so all four models start from checkpoints pretrained on
millions of modern images and are adapted to period damage:

| Task | Model (HF checkpoint) | Params | Adapted | Optimiser / LR | Loss |
|---|---|---|---|---|---|
| Denoise | SCUNet (`deepinv/scunet`) | 17.9 M | all weights | AdamW 2e-5, cosine | Charbonnier |
| Super-res ×4 | Swin2SR (`caidas/swin2SR-realworld-sr-x4-64-bsrgan-psnr`) | 12.1 M | all weights | AdamW 2e-5, cosine | L1 |
| Colorize | DDColor ConvNeXt-L (`piddnad/ddcolor_modelscope`) | 227.9 M | all weights | AdamW 1e-5, cosine | L1 + 0.5 LPIPS + 0.5 colourfulness gap |
| Inpaint | SD 1.5 Inpainting (`stable-diffusion-v1-5/stable-diffusion-inpainting`) | 860 M (3.2 M LoRA) | LoRA r = 16 | AdamW 1e-4, cosine | ε-prediction MSE |

The low learning rates *adapt* the pretrained priors without overwriting them (no
catastrophic forgetting). LoRA freezes SD's weights $W$ and learns a low-rank update
$W' = W + \tfrac{\alpha}{r} BA$ with $B \in \mathbb{R}^{d\times r}, A \in \mathbb{R}^{r\times k}$,
$r \ll \min(d,k)$ (Hu et al., 2021). $B$ is initialised to 0, so step 0 *is* the
pretrained model.

All runs use bf16 mixed precision and gradient accumulation (effective batch 8; 4 for
SD), with validation every 500 steps on a fixed set of 128 validation images (16 for SD,
since each one runs a 20-step diffusion sample). The best checkpoint is selected by PSNR
(denoise, SR) or LPIPS (colorize, inpaint).

In [ ]:
rows = []
for name in ["denoise.yaml", "super_resolution.yaml", "colorize.yaml", "inpaint.yaml"]:
    c = load_config(name).model_dump()
    rows.append({k: c.get(k) for k in ["task", "model_key", "patch_size", "batch_size", "grad_accum", "lr", "max_steps", "val_every"]})
pd.DataFrame(rows).set_index("task")

## 3.1. Denoising: SCUNet

**Architecture.** SCUNet (Zhang et al., 2023) is a U-Net whose blocks split their
channels between a **Swin-Transformer** branch (window self-attention: non-local
self-similarity, which grain removal benefits from) and a **residual convolution**
branch (local detail). It is *blind*: unlike DRUNet it needs no noise-level input,
which suits film grain of unknown strength.

**Objective.** For noisy input $y$ and clean target $x$, minimise the Charbonnier loss
$\mathcal{L} = \frac{1}{N}\sum_i \sqrt{(f_\theta(y)_i - x_i)^2 + \varepsilon^2}$ with
$\varepsilon = 10^{-3}$, a differentiable L1 that is robust to outliers and sharper
than L2.

**Data.** 256 px crops; film grain (spatially correlated, strongest in mid-tones) +
Gaussian σ ∈ [2, 30]/255 + JPEG q ∈ [50, 95] with p = 0.4.

In [ ]:
cfg_denoise = load_config("denoise.yaml")
if RUN_TRAINING:
    run_task(cfg_denoise)
show_run(cfg_denoise.run_name, metrics=("psnr", "ssim", "lpips"), title="Denoising: SCUNet")

## 3.2. Super-resolution ×4: Swin2SR

**Architecture.** Swin2SR (Conde et al., 2022) applies SwinV2 transformer blocks
(scaled cosine attention, log-spaced continuous position bias) in residual groups,
followed by a nearest-neighbour + convolution upsampler. The checkpoint was trained on
BSRGAN's randomised real-world degradations.

**Degradation model.** $y = \big((x * k_\sigma)\!\downarrow_4 + n\big)_{\text{JPEG}}$ with
Gaussian blur $\sigma \in [0.2, 1.6]$, bicubic ×4 downsampling, noise $\le 8/255$ and
JPEG with p = 0.4.

**Objective.** L1 between the 256 px reconstruction and the HR crop. Zero-shot, the
model barely beats bicubic, because its pretraining degradations do not match film
scans. That is the clearest case for fine-tuning.

In [ ]:
cfg_sr = load_config("super_resolution.yaml")
if RUN_TRAINING:
    run_task(cfg_sr)
show_run(cfg_sr.run_name, metrics=("psnr", "ssim", "lpips"), title="Super-resolution ×4: Swin2SR")

## 3.3. Colorization: DDColor

**Architecture.** DDColor (Kang et al., ICCV 2023) has a ConvNeXt-L image encoder
and **dual decoders**: a pixel decoder that upsamples multi-scale features, and a
*colour decoder* whose 100 learnable **colour queries** cross-attend to those features
(a Mask2Former-style transformer). The queries are combined with the pixel features
to predict the CIELAB chroma $\hat{ab}$ for the input lightness $L$.

**Objective.** The output RGB is $\hat{x} = \mathrm{Lab2RGB}(L, \hat{ab})$, implemented
differentiably in PyTorch. We minimise

$\mathcal{L} = \lVert \hat{x} - x \rVert_1 + 0.5\,\mathrm{LPIPS}(\hat{x}, x) + 0.5\,\frac{|C(\hat{x}) - C(x)|}{100}$,

where $C$ is the Hasler–Süsstrunk colourfulness. Colorization is ill-posed (a grey car
could be red or blue), so L1 alone regresses to desaturated averages. The perceptual
and colourfulness terms counteract that.

**Data.** Whole images (global context matters), squashed to 512×512 as DDColor does.
Targets are **fading-corrected** (notebook 01, §3b).

In [ ]:
cfg_color = load_config("colorize.yaml")
if RUN_TRAINING:
    run_task(cfg_color)
show_run(cfg_color.run_name, metrics=("psnr", "lpips", "colorfulness"), title="Colorization: DDColor")

## 3.4. Inpainting: Stable Diffusion 1.5 + LoRA

**Architecture.** Latent diffusion (Rombach et al., 2022). A VAE encodes the image
to a 64×64×4 latent $z_0$. The inpainting UNet takes 9 input channels: the noisy latent
$z_t$, the downsampled mask $m'$, and the latent of the masked image $z_m$. It is
conditioned on CLIP text embeddings $c$.

**Objective (ε-prediction).**
$z_t = \sqrt{\bar\alpha_t} z_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$, $\;t \sim \mathcal{U}\{0,\dots,999\}$,
$\;\mathcal{L} = \lVert \epsilon - \epsilon_\theta([z_t, m', z_m], t, c) \rVert_2^2$.

**Adaptations for photo damage.**
* $m' = \text{maxpool}_8(m)$ rather than nearest-neighbour, so 1–4 px scratches still
  mark their 8×8 latent cell. At inference we dilate the mask by 3 px for generation and
  paste back only the true hole.
* Captions are archive titles ("an old photograph, *title*"), dropped to a generic
  prompt 50 % of the time so the web app works without one.
* Masks: 50 % procedural scratches/cracks/blotches, 50 % NVIDIA irregular masks.

In [ ]:
cfg_inpaint = load_config("inpaint.yaml")
if RUN_TRAINING:
    run_task(cfg_inpaint)
show_run(cfg_inpaint.run_name, metrics=("psnr", "lpips", "psnr_hole"), title="Inpainting: Stable Diffusion 1.5 + LoRA")

## 4. Summary: validation results

Best validation checkpoint per task vs. the classical baseline and the zero-shot model
(fixed validation set). Final, unbiased numbers on the held-out **test** split, plus
real-damage examples, are in notebook 03.

In [ ]:
def summarise(run_name, task, select):
    csv = checkpoint_root() / run_name / "metrics.csv"
    if not csv.is_file():
        return None
    f = pd.read_csv(csv)
    base = f[f["kind"].str.startswith("baseline")].iloc[0]
    zero = f[f["kind"] == "zero_shot"].iloc[0]
    vals = f[f["kind"].isin(["zero_shot", "val"])]
    best = vals.loc[vals[select].idxmax() if select == "psnr" else vals[select].idxmin()]
    return {
        "task": task, "best step": int(best["step"]),
        "PSNR baseline": base["psnr"], "PSNR zero-shot": zero["psnr"], "PSNR fine-tuned": best["psnr"],
        "LPIPS baseline": base["lpips"], "LPIPS zero-shot": zero["lpips"], "LPIPS fine-tuned": best["lpips"],
    }

runs = [(cfg_denoise.run_name, "Denoise", "psnr"), (cfg_sr.run_name, "Super-res ×4", "psnr"),
        (cfg_color.run_name, "Colorize", "lpips"), (cfg_inpaint.run_name, "Inpaint", "lpips")]
summary = pd.DataFrame([r for r in (summarise(*x) for x in runs) if r]).set_index("task")
summary.round(4)